# プロンプトエンジニアリング

プロンプトエンジニアリングでは、モデル呼び出し以前に入力設計と評価設計を固める習慣を作ります。


## 背景と目的

同じモデルでも指示の構造で出力品質が大きく変わります。

プロンプト設計を体系化すると、試行錯誤を減らして再現性を高められます。

入力構造の違いが出力に与える効果を比較します。


## 最初に解きたい疑問

1. 「トークン数をざっくり見積もる」とあるが、実際のトークナイザでは何が違うのか。
2. 制約を増やすと本当に出力が良くなるのか、どんな制約が効くのか。
3. テンプレート化したプロンプトは、どこまで固定してどこを変えるべきか。
4. 検索文脈を入れるとき、何を入れて何を削るべきか。
5. チェック項目やスコアは、実際の品質とどう対応づければいいのか。


## 先に押さえる言葉

- `トークン`: モデルが文章を扱うときの基本単位です。見た目の文字数と一致せず、同じ文章でも分割のされ方が変わります。
- `制約付きプロンプト`: 文字数や出力形式、含める要素を明示して入力する書き方です。曖昧さを減らして出力を揃えやすくします。
- `参考文脈`: 質問に添えて渡す追加情報です。モデルの記憶だけに頼らず、根拠を持った回答を作らせるために使います。
- `簡易評価`: 長さやキーワードの有無のような単純な基準で出力を点検する方法です。厳密な採点ではなく、改善の方向をそろえるために使います。


## 実行前の見取り図

1. `rough_tokens` の値と元の文字列の長さを見て、見積もりがどれくらい粗いかを確認する。
2. `template` の出力を見て、課題・制約・出力形式が1つの入力に整理されているかを確認する。
3. `final_input` の内容を見て、質問と参考文脈の順序が崩れていないかを確認する。


## 出力の読み方

- `checks` の score は長さとキーワードしか見ていないので、良い回答とは限らない。


## つまずきやすい点

- 粗いトークン見積もりを、実際のトークナイザ結果にどう置き換えるかが説明されていない。
- RAG の最小形を、実際の検索器や検索インデックスに拡張する手順が説明されていない。
- テンプレートは作っているが、出力品質との対応がまだ弱い。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- このノートの評価は、実運用の品質評価ではなく簡易ルール一致の確認に近い。


## 検証 1: トークン近似を体験する

最初に、入力長とトークン量の関係を簡易計測します。コスト管理の第一歩です。


In [ ]:
text = '大規模言語モデルは文脈の与え方で応答品質が大きく変わる。'
char_len = len(text)
space_tokens = text.split()
rough_tokens = max(1, char_len // 2)
print('chars=', char_len, 'space_tokens=', len(space_tokens), 'rough_tokens=', rough_tokens)

厳密なトークン化ではありませんが、入力を短く保つ設計感覚を作るには十分です。



## 検証 2: 制約付きプロンプト設計

プロンプトは長く書くより、制約を明確化する方が効果的です。ここではテンプレート化を練習します。


In [ ]:
task = '連鎖律を初学者向けに説明する'
rules = ['120字以内', '式は1つまで', '最後に確認問題を1問']
template = f"課題: {task}\n制約: {'; '.join(rules)}\n出力形式: 段落1つ"
print(template)
print('template_len=', len(template))

この設計は生成APIを呼ばなくても、入力仕様の品質点検として価値があります。



## 定義の確認

1. $p_{\theta}(x_t \mid x_{<t})$
2. $L_{CE} = -\sum_t \log p_{\theta}(x_t \mid x_{<t})$


## 検証 3: 検索文脈を結合する

ここで RAG の最小形を実装します。質問と関連文を結合し、回答入力を作る流れを確認します。


In [ ]:
question = 'ベルマン方程式を高校生向けに説明して'
retrieved = ['価値は将来報酬の割引和で定義する', '現在価値は次状態価値で再帰的に更新できる']
context = '\n'.join(f'- {c}' for c in retrieved)
final_input = f"質問:\n{question}\n\n参考文脈:\n{context}\n\n回答:"
print(final_input)

検索文脈を入れる目的は、モデルの記憶に頼りすぎないことです。根拠付き応答を作りやすくなります。



## 検証 4: 評価項目を数値化する

次に、回答を点検するための簡易スコアを定義します。評価軸を言語化すると改善が継続できます。


In [ ]:
answer = 'ベルマン方程式は、今の価値を次の価値で更新する再帰式です。'
checks = {'length_ok': len(answer) <= 120, 'has_keyword': '価値' in answer, 'has_recurrence': '再帰' in answer}
score = sum(1 for v in checks.values() if v) / len(checks)
print('checks=', checks)
print('score=', round(score, 3))

このような軽量評価でも、改善方向を揃える効果があります。実務ではこの評価軸をチームで共有します。



## 検証 5: 推論コストを見積もる

最後に、入力と出力の長さから概算コストを計算します。モデル選択は性能だけでなく費用とのバランスが必要です。


In [ ]:
input_tokens = 320
output_tokens = 180
price_per_1k = 0.0012
cost = (input_tokens + output_tokens) / 1000 * price_per_1k
print('estimated_cost=', round(cost, 6))

この見積もりを運用前に作ると、スケール時の予算超過を防ぎやすくなります。



## まとめ

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. プロンプトだけで全問題を解決しようとする
2. 評価指標を決めずに改善を繰り返す
3. コストと品質のバランスを見ない
